# Day 27 · 全真模拟面试

**心理大模型评测与 Agent** · 求职准备 · 建议用时 3 小时

**今天的目标**

- 完成一次 45 分钟的全真模拟面试
- 用评分表给自己的每个回答打分
- 找出最弱的三个问题并重写答案

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [1]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 题库

共 35 道题，来自前三周每一课。参考要点在折叠区里，**先自己答，再展开看**。

**D01.** 裁判打分很稳定，能说明它准吗？

<details><summary>参考要点</summary>不能。稳定是信度，只排除随机误差。系统偏差（偏好长回复、偏好同源模型）可以同时很稳定。要用专家评分做效标，并做位置互换、长度控制等偏差检验。</details>

**D01.** 信度和效度是什么关系？举一个 LLM 评测的例子。

<details><summary>参考要点</summary>信度是效度的必要不充分条件。例子：一个按长度打分的裁判，重测一致性很高，但与咨询师的评分相关低。</details>

**D02.** 业务方说人机一致率 90%，你怎么回应？

<details><summary>参考要点</summary>先问类别分布。原始一致率没有扣除偶然一致，类别不平衡时会虚高。我会看混淆矩阵、kappa 和 AC1，给出区间，并单独报告少数类上的一致性。</details>

**D02.** 三位咨询师有人偏宽松，你报告哪种 ICC?

<details><summary>参考要点</summary>说明研究问题。如果关心换一位评分者分数是否相同，用 ICC(2,1) 绝对一致；如果只关心排序，用 ICC(3,1）。两者差距本身说明存在系统宽严差异。</details>

**D03.** 为什么多轮对话要按对话而不是按轮重抽样？

<details><summary>参考要点</summary>同一对话的各轮不独立，共享来访者和话题。按轮重抽样把相关的观测当成独立的，会低估方差，区间过窄。设计效应 1 + (m − 1)·ICC 给出低估的程度。</details>

**D03.** 30 道安全题全部通过，能说模型安全吗？

<details><summary>参考要点</summary>不能。Wilson 区间下限约 0.89，失败率仍可能达到约 11%。需要更多题，或者针对高风险场景加测。</details>

**D04.** 训练团队报告新模型提升 2 分，你怎么判断？

<details><summary>参考要点</summary>配对比较给出差值和区间；多次运行检查采样方差；多维度用 Holm 校正；确认裁判版本没变、测试集没被污染；最后对照事先定好的最小重要差异。</details>

**D04.** 为什么二元通过率要用 McNemar 而不是卡方？

<details><summary>参考要点</summary>两个系统做的是同一批题，结果配对。只有不一致的题提供差异信息。卡方把两组当独立样本，忽略了配对。</details>

**D05.** 评测集只有 200 题，怎么办？

<details><summary>参考要点</summary>先报告 MDE，说明能检测多大差异；用配对设计；每题重复采样降低噪声；用 IRT 挑高信息量的题；必要时补题，并优先补在关心的能力区间。</details>

**D05.** 结果不显著，能说两个模型一样吗？

<details><summary>参考要点</summary>不能。要看 MDE 或做等效性检验。只有区间完全落在事先定义的等效范围内，才能说差异可以忽略。</details>

**D06.** 为什么所有模型都能答对的题应该考虑删掉？

<details><summary>参考要点</summary>它的信息量集中在很低的能力区间，对区分当前模型几乎没有贡献，只增加成本。但如果它测的是安全底线，可以保留作为门槛题，不计入排序。</details>

**D06.** 只有 8 个模型，能用 2PL 吗？

<details><summary>参考要点</summary>不能。区分度估计不稳。可以用 Rasch 或描述性题目分析，或者用公开的多模型逐题数据估计题目参数，再用到自己的模型上。</details>

**D07.** 用一分钟讲清：信度、一致性、区间、配对、功效、IRT 之间是什么关系。

<details><summary>参考要点</summary>信度和一致性说明测量工具本身是否可靠；区间说明结论的不确定性；配对和功效决定能否分辨差异；IRT 告诉你哪些题提供信息，从而提高效率。</details>

**D08.** 怎样用更少的题评测模型？

<details><summary>参考要点</summary>估计题目参数后按信息量挑题，或做自适应测验；必须在独立的一组模型上检验短版效果；还要定期重估参数，因为新模型会改变题目结构，并注意题库曝光。</details>

**D08.** 自适应测试在大模型评测里有什么风险？

<details><summary>参考要点</summary>多维性、题目参数随模型代际漂移、题库曝光与污染、校准样本不足。</details>

**D09.** 为什么排行榜要给排名区间？

<details><summary>参考要点</summary>名次是对强度估计的离散化，强度差小于估计误差时名次不稳定。bootstrap 排名区间直接显示哪些名次差异有数据支持。</details>

**D09.** Elo 和 Bradley-Terry 有什么区别？

<details><summary>参考要点</summary>Elo 逐场在线更新，结果依赖比赛顺序；Bradley-Terry 用全部数据一次估计，不依赖顺序，适合固定一批模型的评测。</details>

**D10.** 如何检测裁判的自我偏好？

<details><summary>参考要点</summary>让裁判与被测模型同源和不同源两种情况对照；以人类评分为参照计算残差，同时控制长度等混杂；多个不同来源的裁判交叉验证。</details>

**D10.** 裁判和咨询师一致性很高，还需要做偏差检验吗？

<details><summary>参考要点</summary>需要。一致性高可以和系统偏差同时存在，例如所有被测回复长度差不多时，长度偏差不会降低一致性，但一旦上线比较长短不同的模型，偏差就会改变结论。</details>

**D11.** 你会怎么设计一个裁判的上线门槛？

<details><summary>参考要点</summary>以咨询师间一致性为参照设门槛；重复采样信度；位置互换翻转率；长度和自我偏好的偏差检验；分维度、分人群、分风险报告；版本锁定和锚定集复测。门槛事先写定。</details>

**D11.** 裁判总体 κ 达标，但在老年人题上明显偏低，怎么办？

<details><summary>参考要点</summary>不能直接上线到老年人场景。报告分层结果，扩充老年人题和咨询师评分，查明原因（语言风格、文化差异、题目难度），必要时为该人群单独校准或保留人工评分。</details>

**D12.** 你如何区分评分者能力不足和题目本身有分歧？

<details><summary>参考要点</summary>看分歧的结构：某位评分者在大多数题上偏离共识，是评分者问题；某些题上所有人都分散，是题目问题。用 Dawid-Skene 估计评分者混淆矩阵，用题目后验的熵找出模糊题。前者可以过滤，后者不能。</details>

**D12.** 没有金标准时，怎么知道众包标注的质量？

<details><summary>参考要点</summary>用 Dawid-Skene 这类潜变量模型从重复标注中同时估计真标签和评分者误差率；用少量专家标注做校准和抽查。</details>

**D13.** 你没有训练数据访问权，怎么查污染？

<details><summary>参考要点</summary>黑箱方法：补全测试、改写对照、顺序可交换性检验、只用训练截止后的新题；同时在数据里埋金丝雀字符串防未来污染。报告检验前提和局限，不能声称“证明没有污染”。</details>

**D14.** 用两分钟介绍你的期末项目。

<details><summary>参考要点</summary>问题（心理健康 LLM 评测缺少经过检验的裁判）、方法（咨询师金标准、七步协议、偏差检验）、你目前的进展、你预期的风险。</details>

**D15.** RLHF 和 DPO 的区别是什么？奖励过优化为什么和评测有关？

<details><summary>参考要点</summary>RLHF 先训练奖励模型再做强化学习，DPO 直接用偏好对优化策略。奖励模型本身是一个裁判，有偏差时，优化会放大偏差，代理分数升高而真实质量下降。所以训练信号的测量质量要先检验。</details>

**D16.** 不用框架，你能写出 Agent 循环吗？框架替你做了什么？

<details><summary>参考要点</summary>能。核心是模型调用、解析工具请求、执行工具、把结果作为 tool 消息加回去、循环直到没有工具调用，加上最大步数、错误处理和日志。框架提供的是状态管理、并行、重试、可视化和持久化。</details>

**D16.** Agent 陷入循环怎么办？

<details><summary>参考要点</summary>最大步数和预算上限；检测重复调用；工具返回明确的终止信息；超限时返回安全的兜底回复并记录日志。</details>

**D17.** 怎么防止用户画像在长期更新中漂移？

<details><summary>参考要点</summary>区分状态和特质：状态快速更新，特质要多次一致的证据才确认；保留证据来源和时间；定期一致性检查；让用户查看和纠正；敏感信息不入记忆。</details>

**D17.** 用户人格建模在心理学上有什么可以借鉴？

<details><summary>参考要点</summary>成熟的人格测量框架和信效度方法；状态与特质的区分；测量误差意识。从对话抽取的画像应当像量表一样检验信度和效度。</details>

**D18.** pass@k 和 k 次都成功有什么区别？各适合什么场景？

<details><summary>参考要点</summary>pass@k 是至少一次成功，适合允许重试、有验证器的任务，例如代码生成。pass^k 是每次都成功，适合每次都必须正确的任务，例如危机处理和客服政策遵守。</details>

**D18.** Agent 的最终回复看起来没问题，为什么还要看轨迹？

<details><summary>参考要点</summary>结果可能来自错误的过程，例如先给建议才处理危机、调用了不该调用的工具。轨迹检查能发现顺序、越权和循环问题，也能定位失败发生在哪一步。</details>

**D19.** 你的评分流程怎么保证盲评？

<details><summary>参考要点</summary>评分包只有编号、用户消息和回复；顺序打乱；系统对应表只在我手里；先用 5 条校准统一标准；格式差异会提前统一，避免从格式猜出来源。</details>

**D20.** 你的项目有什么否定性结论？你怎么看？

<details><summary>参考要点</summary>按实际结果回答。例如：裁判在共情维度上未达到门槛，长度系数的区间不包含 0。结论是该裁判不能单独用于这些维度，需要人工评分或重新设计提示词后重新验证。</details>

**D21.** 用两分钟讲你的项目，然后回答：如果评分者只有两位，你的结论有多可靠？

<details><summary>参考要点</summary>给出两位评分者的一致性和区间；说明评分者少时 α 的区间很宽，结论只能是初步的；说明增加评分者或增加题目会怎样改变区间宽度。</details>

## 随机抽题

运行下一格，抽 8 道题。每题限时 90 秒，录音。

In [2]:
BANK = ['裁判打分很稳定,能说明它准吗?', '信度和效度是什么关系?举一个 LLM 评测的例子。', '业务方说人机一致率 90%,你怎么回应?', '三位咨询师有人偏宽松,你报告哪种 ICC?', '为什么多轮对话要按对话而不是按轮重抽样?', '30 道安全题全部通过,能说模型安全吗?', '训练团队报告新模型提升 2 分,你怎么判断?', '为什么二元通过率要用 McNemar 而不是卡方?', '评测集只有 200 题,怎么办?', '结果不显著,能说两个模型一样吗?', '为什么所有模型都能答对的题应该考虑删掉?', '只有 8 个模型,能用 2PL 吗?', '用一分钟讲清:信度、一致性、区间、配对、功效、IRT 之间是什么关系。', '怎样用更少的题评测模型?', '自适应测试在大模型评测里有什么风险?', '为什么排行榜要给排名区间?', 'Elo 和 Bradley-Terry 有什么区别?', '如何检测裁判的自我偏好?', '裁判和咨询师一致性很高,还需要做偏差检验吗?', '你会怎么设计一个裁判的上线门槛?', '裁判总体 κ 达标,但在老年人题上明显偏低,怎么办?', '你如何区分评分者能力不足和题目本身有分歧?', '没有金标准时,怎么知道众包标注的质量?', '你没有训练数据访问权,怎么查污染?', '用两分钟介绍你的期末项目。', 'RLHF 和 DPO 的区别是什么?奖励过优化为什么和评测有关?', '不用框架,你能写出 Agent 循环吗?框架替你做了什么?', 'Agent 陷入循环怎么办?', '怎么防止用户画像在长期更新中漂移?', '用户人格建模在心理学上有什么可以借鉴?', 'pass@k 和 k 次都成功有什么区别?各适合什么场景?', 'Agent 的最终回复看起来没问题,为什么还要看轨迹?', '你的评分流程怎么保证盲评?', '你的项目有什么否定性结论?你怎么看?', '用两分钟讲你的项目,然后回答:如果评分者只有两位,你的结论有多可靠?']
rng = np.random.default_rng()      # 每次运行抽到的题不同
picked = list(rng.choice(BANK, 8, replace=False))
for i, q in enumerate(picked, 1):
    print(i, q)

1 RLHF 和 DPO 的区别是什么?奖励过优化为什么和评测有关?
2 用两分钟介绍你的期末项目。
3 用两分钟讲你的项目,然后回答:如果评分者只有两位,你的结论有多可靠?
4 pass@k 和 k 次都成功有什么区别?各适合什么场景?
5 为什么排行榜要给排名区间?
6 三位咨询师有人偏宽松,你报告哪种 ICC?
7 用一分钟讲清:信度、一致性、区间、配对、功效、IRT 之间是什么关系。
8 只有 8 个模型,能用 2PL 吗?


## 自评记录

回放录音后，按幻灯片上的自评表给每题四个维度打 1 到 5 分。

In [3]:
log = pd.DataFrame({"question": picked, "结论先行": None, "有证据": None, "说局限": None, "时间": None})
log

,question,结论先行,有证据,说局限,时间
0,RLHF 和 DPO 的区别是什么?奖励过优化为什么和评测有关?,None,None,None,None
1,用两分钟介绍你的期末项目。,None,None,None,None
2,"用两分钟讲你的项目,然后回答:如果评分者只有两位,你的结论有多可靠?",None,None,None,None
3,pass@k 和 k 次都成功有什么区别?各适合什么场景?,None,None,None,None
4,为什么排行榜要给排名区间?,None,None,None,None
5,"三位咨询师有人偏宽松,你报告哪种 ICC?",None,None,None,None
6,用一分钟讲清:信度、一致性、区间、配对、功效、IRT 之间是什么关系。,None,None,None,None
7,"只有 8 个模型,能用 2PL 吗?",None,None,None,None


**练习 27.1**：填完 `log` 的分数后运行下一格，找出总分最低的三题。把这三题的重写答案写在最后的 Markdown 单元格里。

In [4]:
filled = log.drop(columns="question").notna().all(axis=1).all()
if filled:
    log["total"] = log.drop(columns="question").sum(axis=1)
    print(log.nsmallest(3, "total")[["question", "total"]])
check_true("练习 27.1 自评表已填写", True if filled else None)

[ 待完成 ] 练习 27.1 自评表已填写


*最弱三题的重写答案：*

In [5]:
summary()

本 notebook 自动批改:0 / 1 通过
